<a href="https://colab.research.google.com/github/Lizandraferrari/processamento-de-linguagem-neural/blob/roteiro-4/Roteiro_de_Pr%C3%A1ticas_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install kagglehub[pandas-datasets]
!pip install -U spacy
!python -m spacy download pt_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 50.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [34]:
import kagglehub
import pandas as pd
import os

#importei daqui https://www.kaggle.com/datasets/ddmasterdon/anime-2023-with-reviews/
# Download latest version
path = kagglehub.dataset_download("satkar003/anime-reviews")
print("Path to dataset files:", path)
csv_file = os.path.join(path, 'review_0_1.csv')

# Carregar CSV
df = pd.read_csv(csv_file)
dados = df['summary'].astype(str).head(500)

Using Colab cache for faster access to the 'anime-reviews' dataset.
Path to dataset files: /kaggle/input/anime-reviews


In [36]:
import re

# Aplica a substituição regex a cada elemento da Series 'dados'
texto_limpo = dados.apply(lambda x: re.sub(r'[^A-Za-zÀ-ÿ\s]', '', x))
# r'[^A-Za-zÀ-ÿ\s]'>>> expressao regular que define um conjunto de caracteres a serem removidos
# '' substitui a expressão regular por uma string vazia

# Aplica a conversão para minúsculas a cada elemento da Series 'texto_limpo'
texto_normalizado = texto_limpo.str.lower()

print(f'Primeiros 5 textos originais: ')
print(dados.head())
print(f'\nPrimeiros 5 textos limpos: ')
print(texto_limpo.head())
print(f'\nPrimeiros 5 textos normalizados: ')
print(texto_normalizado.head())

Primeiros 5 textos originais: 
0    This anime has just not got going for me perso...
1    This anime feels like a typical Cinderella sto...
2    If you’re into the whole shota/Onee-san dynami...
3    Started watching this at 3 am and got done at ...
4    Dragon Ball Daima is a show I wanted to like, ...
Name: summary, dtype: object

Primeiros 5 textos limpos: 
0    This anime has just not got going for me perso...
1    This anime feels like a typical Cinderella sto...
2    If youre into the whole shotaOneesan dynamic M...
3    Started watching this at  am and got done at  ...
4    Dragon Ball Daima is a show I wanted to like b...
Name: summary, dtype: object

Primeiros 5 textos normalizados: 
0    this anime has just not got going for me perso...
1    this anime feels like a typical cinderella sto...
2    if youre into the whole shotaoneesan dynamic m...
3    started watching this at  am and got done at  ...
4    dragon ball daima is a show i wanted to like b...
Name: summary, dtype: 

In [37]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt_tab')

tokens = texto_normalizado.apply(word_tokenize)

print(f'Primeiros 5 textos originais: {dados.head().tolist()}')
print(f'Primeiros 5 textos limpos: {texto_limpo.head().tolist()}')
print(f'Primeiros 5 textos normalizados: {texto_normalizado.head().tolist()}')
print(f'Primeiros 5 tokens extraidos: {tokens.head().tolist()}\n')


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Primeiros 5 textos originais: ['This anime has just not got going for me personally. Like a majority of the animes that I have watched, I have never read the manga overall. I went into this anime blind as a bat, and my expectations were less than stellar. From episodes one to five, it felt like it was the same plot over and over again, Sure you get a character here and there for the plot but it feels like it is a very mundane story, that has the saving grace of an opening and ending songs doing more to make up for a lack of structure. When I read the                  ...plot of Call of the Night, I thought it would be intense and have this result of character building. It falls flat like a three day Pepsi from a two liter bottle that someone forgot to put back in the fridge after a party. I am twelve episodes in and this anime is a solid five out of ten for me.', 'This anime feels like a typical Cinderella story but with strange powers that seem completely unnecessary. The inclusion of

In [38]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

def extrair_vocabulario(docs):
    """
    Transforma uma lista de frases em uma lista ordenada de termos únicos.
    Foco: Normalização e Unicidade.
    """
    # 1. Normalização e Tokenização em uma única lista
    todos_os_tokens = []
    for frase in docs:

        tokens_frase = frase.lower().split()  # Tokenização bruta por espaço
        todos_os_tokens.extend(tokens_frase)

    # 2. Garantia de Unicidade usando Set (Estrutura de Hash)
    vocab_set = set(todos_os_tokens)

    # 3. Ordenação Alfabética para consistência de índices
    vocab_ordenado = sorted(list(vocab_set))

    return vocab_ordenado

#tem que ajusar para ler em string
corpus = tokens.apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))

# Execução
vocabulario = extrair_vocabulario(corpus.tolist())
print(f"Vocabulário Ordenado ({len(vocabulario)} termos):")
print(vocabulario[:10])


Vocabulário Ordenado (12223 termos):
['a', 'aaaaaah', 'abandon', 'abandoned', 'abandoning', 'abd', 'abduct', 'abductions', 'abilities', 'ability']
...


In [39]:
def construir_matriz_densa(docs, vocab):
    """
    Cria uma Matriz Termo-Documento (D x V) usando NumPy.
    Demonstra a alocação de memória para elementos nulos (zeros).
    """
    n_docs = len(docs)
    n_vocab = len(vocab)
    matriz = np.zeros((n_docs, n_vocab), dtype=np.int64)

    for i, frase in enumerate(docs):
        tokens_frase = frase.lower().split()
        for j, termo in enumerate(vocab):
            # Contagem de frequência
            matriz[i][j] = tokens_frase.count(termo)

    return matriz


matriz_final = construir_matriz_densa(corpus.tolist(), vocabulario)
print("Matriz Termo-Documento (Representação Densa NumPy):")
print(matriz_final)


Matriz Termo-Documento (Representação Densa NumPy):
[[ 9  0  0 ...  0  0  0]
 [ 1  0  0 ...  0  0  0]
 [10  0  0 ...  0  0  0]
 ...
 [14  0  0 ...  0  0  0]
 [ 4  0  0 ...  0  0  0]
 [12  0  0 ...  0  0  0]]


In [40]:
# 1. Vetorização Profissional
vectorizer = CountVectorizer()
X_esparso = vectorizer.fit_transform(corpus)

# 2. Conversão para Denso apenas para fins de comparação didática
X_denso = X_esparso.toarray()

# 3. Prova Matemática de Desperdício (RAM)
# Para matrizes esparsas, somamos os arrays de dados, índices e ponteiros
memoria_esparsa = X_esparso.data.nbytes + X_esparso.indptr.nbytes + X_esparso.indices.nbytes
memoria_densa = X_denso.nbytes

# 4. Cálculo de Esparsidade
total_celulas = X_denso.size
zeros = total_celulas - np.count_nonzero(X_denso)
percentual_zeros = (zeros / total_celulas) * 100

print(f"--- Relatório de Performance de Memória ---")
print(f"Memória RAM (Matriz Densa): {memoria_densa} bytes") #tentei realizar utilizando maior quantidade de linhas mas a memória estourou conforme o informado no documento da aula. são aproximadamente 50mil linhas!
print(f"Memória RAM (Matriz Esparsa): {memoria_esparsa} bytes")
print(f"Eficiência de Memória: {((memoria_densa - memoria_esparsa) / memoria_densa) * 100:.2f}% de economia")
print(f"Porcentagem de Esparsidade: {percentual_zeros:.2f}% de zeros na matriz")


--- Relatório de Performance de Memória ---
Memória RAM (Matriz Densa): 48796000 bytes
Memória RAM (Matriz Esparsa): 987024 bytes
Eficiência de Memória: 97.98% de economia
Porcentagem de Esparsidade: 98.65% de zeros na matriz
